# 06B PHA (CGenFF) + enzyme in water, GROMACS

**For enzyme systems:** run 05B first (CGenFF parameters from the CHARMM-GUI website), then this notebook. `06_optional_gaff2_openmm_pha_enzyme_in_water.ipynb` is another way, using OpenMM.

CHARMM36m protein · CGenFF polymer · CHARMM TIP3P water + SOD/CLA ions · GROMACS.

This notebook turns a CHARMM-GUI **Solution Builder** GROMACS download of an enzyme + polymer complex in water into a run folder and checks it before any simulation. For a PHA alone in water, use `06A_gaff2_gromacs_pha_in_water.ipynb`.

**The enzyme–PHA production simulations used this route.** For example, GK13_P3HO_4 was built from CHARMM-GUI job 8214536317 (docking pose `GK13_PHO4_pose2_complex.pdb`); a folder prepared here from that download is byte-identical to the production input.

## How the run folder is made

This follows the steps used for the production runs:

1. Copy only the download's `gromacs/` folder (`topol.top`, `toppar/`, `index.ndx`, coordinates) into a new run folder. The download is never modified.
2. Rename `step3_input.gro` to `step5_input.gro`, the name the run scripts expect.
3. Leave out CHARMM-GUI's own `step4.x`/`step5` mdp files.
4. Add the CHARMM/GROMACS run files: `step6.0_minimization.mdp` (identical to CHARMM-GUI's `step4.0`), `step6.1_nvt.mdp` (125 ps NVT), `step6.2_npt.mdp` (500 ps NPT), `step7_production.mdp` (200 ns), `run_step6_local.sh` and `run_hpc_equilibration_production.slurm`.

The run files come from `src/iphasimulator/data/charmm_gromacs/` (the production set). Set `TEMPLATE_DIR` to use your own folder instead.

## 1. Inputs

- `SOLUTION_BUILDER_DIR`: the unpacked Solution Builder download of the enzyme + polymer complex (the folder that contains `gromacs/`), or its `gromacs/` folder. It is only read.
- `RUN_DIR`: a **new** folder for the run. It must not exist yet and must not be inside the download.
- `SDF_PATH`: the SDF uploaded to CHARMM-GUI, for bond orders in the stereocentre check (optional; the check is skipped without it).
- `TEMPLATE_DIR`: `None` for the packaged CHARMM/GROMACS run files, or a folder with your own.
- `JOB_NAME`: the SLURM job name.

In [ ]:
from pathlib import Path
import sys

repo_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "iphasimulator").exists())
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))
from iphasimulator.charmmgui_import import (
    format_checks,
    prepare_charmm_gui_run_folder,
    read_minimization_result,
    validate_charmm_gui_run_folder,
)
from iphasimulator.simulation_gromacs_runner import run_gromacs_local_minimization

SOLUTION_BUILDER_DIR = None  # e.g. Path("/path/to/GK13_PHO4_charmm-gui-XXXXXXXXXX")
RUN_DIR = None               # e.g. Path("/path/to/new/GK13_P3HO_4_gromacs")
SDF_PATH = None              # e.g. repo_root / "examples/output/polymer_structures/PHO4_R.sdf"
TEMPLATE_DIR = None
JOB_NAME = "charmm_gromacs"

## 2. Prepare the run folder

In [ ]:
if SOLUTION_BUILDER_DIR is None or RUN_DIR is None:
    print("Set SOLUTION_BUILDER_DIR and RUN_DIR in step 1 to prepare the run folder.")
else:
    run_dir = prepare_charmm_gui_run_folder(
        SOLUTION_BUILDER_DIR, RUN_DIR, template_dir=TEMPLATE_DIR, job_name=JOB_NAME
    )
    print("Prepared:", run_dir)
    for path in sorted(run_dir.iterdir()):
        print("  ", path.name + ("/" if path.is_dir() else ""))

## 3. Read-only checks

- `[ defaults ]` is CHARMM-style (`1 2 yes 1.0 1.0`) and every `#include` file exists.
- `step5_input.gro` has as many atoms as the topology.
- Molecule counts (LIG, protein chains, TIP3, SOD, CLA) and a net charge of about 0.
- LIG atom types are CGenFF-style (e.g. CG331, OG2D1) and LIG atoms are in the same order as `LIG.itp`.
- Every LIG stereocentre is still R in the coordinates GROMACS will use.
- All run files are present and `index.ndx` has the `SOLU`/`SOLV` groups the mdp files use.

The second cell checks the system type: an enzyme + polymer package must contain a protein chain, the polymer (`LIG`), water and ions.

In [ ]:
if RUN_DIR is None or not Path(RUN_DIR).is_dir():
    print("Prepare the run folder in step 2 first.")
else:
    print(format_checks(validate_charmm_gui_run_folder(RUN_DIR, sdf_path=SDF_PATH)))

In [ ]:
from iphasimulator.charmmgui_import import ION_MOLECULES, WATER_MOLECULES, read_topology_molecule_counts

if RUN_DIR is None or not (Path(RUN_DIR) / "topol.top").is_file():
    print("Prepare the run folder in step 2 first.")
else:
    counts = read_topology_molecule_counts(Path(RUN_DIR) / "topol.top")
    names = {name.upper(): count for name, count in counts.items()}
    protein = [name for name in counts if name.upper().startswith("PRO")]
    is_enzyme_polymer_in_water = (
        bool(protein)
        and names.get("LIG", 0) >= 1
        and any(names.get(water, 0) for water in WATER_MOLECULES)
        and any(names.get(ion, 0) for ion in ION_MOLECULES)
    )
    print(f"[{'PASS' if is_enzyme_polymer_in_water else 'FAIL'}] enzyme + polymer in water (protein + LIG + water + ions): {counts}")

## 4. Optional: local minimisation

Runs `bash run_step6_local.sh` in `RUN_DIR` (`grompp` + `mdrun` for `step6.0_minimization`, with the backbone/side-chain restraints from CHARMM-GUI). For an enzyme–polymer box of about 140 000 atoms this takes minutes on a laptop.

In [ ]:
RUN_LOCAL_MINIMIZATION = False

if not RUN_LOCAL_MINIMIZATION:
    print("Set RUN_LOCAL_MINIMIZATION = True to run step 6.0 locally.")
else:
    run_gromacs_local_minimization(RUN_DIR)
    result = read_minimization_result(Path(RUN_DIR) / "step6.0_minimization.log")
    print(f"Potential energy: {result.potential_energy} kJ/mol")
    print(f"Maximum force:    {result.maximum_force} kJ/mol/nm")

## 5. Hand-off to 07

`07_hpc_execution.ipynb` (CHARMM/GROMACS section) needs the run folder with `step6.0_minimization.gro` from step 4 and `run_hpc_equilibration_production.slurm`, which runs NVT (6.1), NPT (6.2) and production (7) on the cluster. The SLURM module line is for the KCL GPU cluster; adapt it for other systems.

In [ ]:
if RUN_DIR is None:
    print("No run folder yet.")
else:
    run_dir = Path(RUN_DIR)
    for name in ("step6.0_minimization.gro", "run_hpc_equilibration_production.slurm", "index.ndx", "topol.top"):
        print(f"{'found  ' if (run_dir / name).exists() else 'missing'}  {name}")
    print("On the cluster:")
    print(f"  cd {run_dir}")
    print("  sbatch run_hpc_equilibration_production.slurm")